# Syntext: Large-Scale Bigram Weight Table Generator

Generates `src/tokenizer/weights.rs` from
[`bigcode/the-stack-dedup`](https://huggingface.co/datasets/bigcode/the-stack-dedup)
(3 TB+, deduplicated, multilingual, permissively licensed).

**This version uses bulk Parquet download instead of streaming.**
The previous streaming approach ran at ~3 MB/s (Python iterator overhead
over HTTP). Bulk download hits 50–200 MB/s from the HuggingFace CDN,
and vectorized `np.bincount` / CuPy replaces per-file Python loops.
Result: **100 GB in ~20–30 min** instead of ~9 hours.

**Before running:**
1. Visit https://huggingface.co/datasets/bigcode/the-stack-dedup
2. Click **"Agree and access repository"**
3. Set Colab runtime to **GPU** (Runtime → Change runtime type → T4/A100)
4. Add your HuggingFace token to Colab secrets (🔑 sidebar → `HF_TOKEN`)

## 1. Install dependencies + detect GPU

In [ ]:
!pip install -q datasets huggingface_hub numpy pyarrow

import shutil, subprocess

GPU = False
if shutil.which("nvidia-smi"):
    try:
        !pip install -q cupy-cuda12x 2>/dev/null
        import cupy as cp
        name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode()
        mem_gb = cp.cuda.Device(0).mem_info[1] / 1e9
        GPU = True
        print(f"GPU: {name} ({mem_gb:.0f} GB)")
    except Exception as e:
        print(f"GPU detected but CuPy failed: {e}")
        print("Falling back to CPU (still fast with bulk download).")

if not GPU:
    print("No GPU. Using CPU with np.bincount (still 10-50x faster than streaming).")

## 2. Authenticate

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("Authenticated via Colab secrets.")
except Exception:
    print("HF_TOKEN not in Colab secrets. Manual login:")
    login()

## 3. Configuration

In [ ]:
import numpy as np
import pyarrow.parquet as pq
import os, time, logging, warnings
from pathlib import Path

logging.getLogger("datasets.load").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message=".*trust_remote_code.*")

# ── Target ─────────────────────────────────────────────────────────────────
# 100 GB recommended. At bulk download speeds, takes ~20-30 min.
# For a quick test: 2_000_000_000 (2 GB, ~2 min).
TARGET_BYTES = 100 * 1_000_000_000

DATASET_NAME = "bigcode/the-stack-dedup"

# the-stack-dedup shard names. C++ is "cpp" not "c++".
LANGUAGES = {
    # Tier 1 — agent-heavy
    "python": 2.0, "javascript": 2.0, "typescript": 2.0, "rust": 2.0,
    "go": 1.5, "java": 1.5, "c": 1.5, "cpp": 1.5,
    # Tier 2 — broader coverage
    "c-sharp": 1.0, "ruby": 1.0, "php": 1.0,
    "swift": 0.8, "kotlin": 0.8,
    "scala": 0.7, "shell": 0.7,
    "lua": 0.5, "haskell": 0.5, "r": 0.5,
    "perl": 0.4, "dart": 0.4,
    "elixir": 0.3, "julia": 0.3, "ocaml": 0.3, "zig": 0.3,
}

MAX_FILE_BYTES = 500_000       # Skip files > 500 KB (minified JS, generated)
PROCESS_CHUNK_MB = 256         # Max MB to concat before counting (bounds RAM)
DOWNLOAD_DIR = "/content/hf_shard"  # Temp dir for each shard, deleted after
CHECKPOINT_PATH = "/content/bigram_checkpoint_v2.npz"
OUTPUT_PATH = "/content/weights.rs"
BOUNDARY_THRESHOLD = 28000     # syntext's gram boundary threshold

print(f"Target:  {TARGET_BYTES / 1e9:.0f} GB | {len(LANGUAGES)} languages")
print(f"Dataset: {DATASET_NAME}")
print(f"GPU:     {'Yes (' + str(GPU) + ')' if GPU else 'No (CPU)'}")
print(f"Chunk:   {PROCESS_CHUNK_MB} MB per batch")

## 4. Validate access + list shards

In [ ]:
from huggingface_hub import HfFileSystem, hf_hub_download

fs = HfFileSystem()

def list_shards(dataset_name, lang):
    """List Parquet shard filenames for a language."""
    prefix = f"datasets/{dataset_name}/data/{lang}"
    try:
        files = fs.ls(prefix, detail=False)
        shards = sorted([f.split("/")[-1] for f in files if f.endswith(".parquet")])
        return shards
    except Exception as e:
        return []


# ── Probe access ───────────────────────────────────────────────────────────
first_lang = next(iter(LANGUAGES))
print(f"Probing {DATASET_NAME} (lang={first_lang})...")
try:
    test_shards = list_shards(DATASET_NAME, first_lang)
    if not test_shards:
        raise RuntimeError("No shards found")
    # Try actually downloading a tiny piece
    test_path = hf_hub_download(
        DATASET_NAME, f"data/{first_lang}/{test_shards[0]}",
        repo_type="dataset", local_dir=DOWNLOAD_DIR,
    )
    test_tbl = pq.read_table(test_path, columns=["content"])
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
    print(f"  Access confirmed. First shard: {test_tbl.num_rows:,} files.")
except Exception as e:
    err = str(e).lower()
    if "gated" in err or "access" in err or "401" in err or "403" in err:
        print("\n" + "=" * 64)
        print(f"  ACCESS DENIED: {DATASET_NAME}")
        print("=" * 64)
        print(f"\n  1. Visit https://huggingface.co/datasets/{DATASET_NAME}")
        print("  2. Click 'Agree and access repository'")
        print("  3. Re-run this cell")
        print("=" * 64)
    raise

# ── List all shards per language ───────────────────────────────────────────
print(f"\nListing shards for {len(LANGUAGES)} languages...")
lang_shards = {}
skipped = []
for lang in LANGUAGES:
    shards = list_shards(DATASET_NAME, lang)
    if shards:
        lang_shards[lang] = shards
    else:
        skipped.append(lang)

total_shards = sum(len(s) for s in lang_shards.values())
print(f"  {len(lang_shards)} languages, {total_shards} total shards")
if skipped:
    print(f"  Skipped (no data): {', '.join(skipped)}")

# Filter LANGUAGES to accessible ones.
LANGUAGES = {k: v for k, v in LANGUAGES.items() if k in lang_shards}

## 5. Processing functions

In [ ]:
def count_pairs_from_bytes(combined: bytes, use_gpu: bool) -> np.ndarray:
    """Count bigram pairs from concatenated, already-lowercased bytes.

    Returns int64 array of length 65536.
    Uses np.bincount (or cp.bincount on GPU) — much faster than np.add.at.
    """
    if len(combined) < 2:
        return np.zeros(65536, dtype=np.int64)

    arr = np.frombuffer(combined, dtype=np.uint8)

    if use_gpu:
        import cupy as cp
        arr_gpu = cp.asarray(arr)
        pairs = (arr_gpu[:-1].astype(cp.int32) << 8) | arr_gpu[1:].astype(cp.int32)
        counts = cp.bincount(pairs, minlength=65536).astype(cp.int64)
        return cp.asnumpy(counts)
    else:
        pairs = (arr[:-1].astype(np.int32) << 8) | arr[1:].astype(np.int32)
        return np.bincount(pairs, minlength=65536).astype(np.int64)


def process_parquet_shard(
    parquet_path: str,
    max_file_bytes: int,
    chunk_mb: int,
    use_gpu: bool,
    byte_limit: int = 0,
) -> tuple[np.ndarray, int, int, int]:
    """Read a Parquet shard and count bigrams.

    Processes in chunks of ~chunk_mb to bound memory.
    Stops early if byte_limit > 0 and we've counted enough.

    Returns (counts[65536], bytes_counted, files_counted, files_skipped).
    """
    table = pq.read_table(parquet_path, columns=["content"])
    contents = table.column("content")

    counts = np.zeros(65536, dtype=np.int64)
    total_bytes = 0
    file_count = 0
    skipped = 0
    chunk_limit = chunk_mb * 1024 * 1024

    # Collect raw bytes in chunks, process each chunk vectorized.
    chunk_parts = []
    chunk_size = 0

    for content in contents.to_pylist():
        if byte_limit > 0 and total_bytes >= byte_limit:
            break

        if content is None or not content:
            continue

        raw = content.encode("utf-8", errors="replace")
        n = len(raw)

        if n > max_file_bytes:
            skipped += 1
            continue
        if n < 2:
            continue

        chunk_parts.append(raw)
        chunk_size += n
        total_bytes += n
        file_count += 1

        # Flush chunk when it reaches the limit.
        if chunk_size >= chunk_limit:
            combined = b"".join(chunk_parts)
            # Vectorized ASCII lowercase.
            arr = np.frombuffer(combined, dtype=np.uint8).copy()
            mask = (arr >= 0x41) & (arr <= 0x5A)
            arr[mask] |= 0x20
            counts += count_pairs_from_bytes(arr.tobytes(), use_gpu)
            chunk_parts = []
            chunk_size = 0

    # Flush remaining.
    if chunk_parts:
        combined = b"".join(chunk_parts)
        arr = np.frombuffer(combined, dtype=np.uint8).copy()
        mask = (arr >= 0x41) & (arr <= 0x5A)
        arr[mask] |= 0x20
        counts += count_pairs_from_bytes(arr.tobytes(), use_gpu)

    return counts, total_bytes, file_count, skipped

## 6. Main loop: download → process → delete → checkpoint

Downloads one Parquet shard at a time from HuggingFace CDN (50–200 MB/s),
processes it, deletes it. Peak disk usage: one shard (~100–500 MB).

Checkpoints after every shard. Re-run to resume.

**To restart from scratch:** `!rm /content/bigram_checkpoint_v2.npz`

In [ ]:
def save_checkpoint(path, counts, total_bytes, lang_bytes, lang_shard_idx):
    langs = list(lang_bytes.keys())
    np.savez_compressed(
        path,
        counts=counts,
        total_bytes=np.array(total_bytes),
        lang_names=np.array(langs),
        lang_bytes_vals=np.array([lang_bytes.get(l, 0) for l in langs]),
        lang_shard_vals=np.array([lang_shard_idx.get(l, 0) for l in langs]),
    )


def load_checkpoint(path):
    if not os.path.exists(path):
        return np.zeros(65536, dtype=np.int64), 0, {}, {}
    ckpt = np.load(path, allow_pickle=False)
    counts = ckpt["counts"]
    total_bytes = int(ckpt["total_bytes"])
    langs = [str(n) for n in ckpt["lang_names"]]
    lang_bytes = dict(zip(langs, [int(b) for b in ckpt["lang_bytes_vals"]]))
    lang_shard_idx = dict(zip(langs, [int(s) for s in ckpt["lang_shard_vals"]]))
    return counts, total_bytes, lang_bytes, lang_shard_idx


# ── Resume or start fresh ──────────────────────────────────────────────────
counts, total_bytes, lang_bytes_done, lang_shard_done = load_checkpoint(CHECKPOINT_PATH)
if total_bytes > 0:
    pct = np.count_nonzero(counts) / 65536 * 100
    print(f"Resumed: {total_bytes / 1e9:.2f} GB, {pct:.1f}% pair coverage")
else:
    print("Starting fresh.")

# Per-language byte targets.
total_weight = sum(LANGUAGES.values())
lang_targets = {
    lang: int(TARGET_BYTES * w / total_weight)
    for lang, w in LANGUAGES.items()
}

wall_start = time.time()
total_files = 0
total_skipped = 0

for lang, byte_target in lang_targets.items():
    already = lang_bytes_done.get(lang, 0)
    if already >= byte_target:
        print(f"  {lang}: done ({already / 1e9:.2f} GB)")
        continue

    shards = lang_shards.get(lang, [])
    start_shard = lang_shard_done.get(lang, 0)
    if start_shard >= len(shards):
        print(f"  {lang}: all {len(shards)} shards processed ({already / 1e9:.2f} GB)")
        continue

    lang_bytes_local = already
    remaining = byte_target - already
    print(f"  {lang}: {remaining / 1e9:.2f} GB remaining, "
          f"shards {start_shard}–{len(shards)-1} of {len(shards)}")

    for shard_idx in range(start_shard, len(shards)):
        if lang_bytes_local >= byte_target:
            break

        shard_name = shards[shard_idx]
        shard_remaining = byte_target - lang_bytes_local

        try:
            # Download.
            t0 = time.time()
            local_path = hf_hub_download(
                DATASET_NAME,
                f"data/{lang}/{shard_name}",
                repo_type="dataset",
                local_dir=DOWNLOAD_DIR,
            )
            dl_sec = time.time() - t0
            dl_mb = os.path.getsize(local_path) / 1e6

            # Process.
            t1 = time.time()
            shard_counts, shard_bytes, shard_files, shard_skipped = process_parquet_shard(
                local_path, MAX_FILE_BYTES, PROCESS_CHUNK_MB, GPU,
                byte_limit=shard_remaining,
            )
            proc_sec = time.time() - t1

            counts += shard_counts
            total_bytes += shard_bytes
            lang_bytes_local += shard_bytes
            total_files += shard_files
            total_skipped += shard_skipped

        except Exception as e:
            print(f"    shard {shard_idx} error: {e}")
            continue
        finally:
            # Always clean up disk.
            shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)

        # Checkpoint after every shard.
        lang_bytes_done[lang] = lang_bytes_local
        lang_shard_done[lang] = shard_idx + 1
        save_checkpoint(CHECKPOINT_PATH, counts, total_bytes, lang_bytes_done, lang_shard_done)

        # Progress.
        elapsed = time.time() - wall_start
        pct = total_bytes / TARGET_BYTES * 100
        coverage = np.count_nonzero(counts) / 65536 * 100
        rate = total_bytes / elapsed / 1e6 if elapsed > 0 else 0
        eta = (TARGET_BYTES - total_bytes) / (total_bytes / elapsed) / 60 if total_bytes > 0 else 0
        print(
            f"    [{shard_idx+1}/{len(shards)}] "
            f"{dl_mb:.0f}MB dl:{dl_sec:.1f}s proc:{proc_sec:.1f}s | "
            f"{total_bytes / 1e9:.1f}GB ({pct:.0f}%) | "
            f"{rate:.0f} MB/s | ETA {eta:.0f}m | "
            f"{coverage:.1f}% coverage"
        )

    print(f"    {lang} done: {lang_bytes_local / 1e9:.2f} GB, {total_files:,} files")

# Final checkpoint.
save_checkpoint(CHECKPOINT_PATH, counts, total_bytes, lang_bytes_done, lang_shard_done)

elapsed = time.time() - wall_start
coverage = np.count_nonzero(counts) / 65536 * 100
print(f"\nFinished: {total_bytes / 1e9:.2f} GB in {elapsed / 60:.1f} min")
print(f"Throughput: {total_bytes / elapsed / 1e6:.0f} MB/s effective")
print(f"Files: {total_files:,} processed, {total_skipped:,} skipped (oversized)")
print(f"Pair coverage: {coverage:.1f}% ({np.count_nonzero(counts):,} / 65,536)")

## 7. Convergence analysis

In [ ]:
def convergence_analysis(counts, total_bytes):
    nonzero = counts > 0
    nz = nonzero.sum()
    total = 65536
    zero = total - nz

    print(f"Coverage: {nz:,} / {total:,} ({nz/total*100:.1f}%)")

    # Break down zero-count pairs.
    ascii_zeros = 0
    ctrl_zeros = 0
    high_zeros = 0
    for idx in range(total):
        if counts[idx] > 0:
            continue
        b1, b2 = idx >> 8, idx & 0xFF
        if 32 <= b1 < 127 and 32 <= b2 < 127:
            ascii_zeros += 1
        elif b1 < 32 or b2 < 32:
            ctrl_zeros += 1
        else:
            high_zeros += 1

    print(f"\nZero-count breakdown ({zero:,} total):")
    print(f"  Printable ASCII: {ascii_zeros:,}  (would improve with more data)")
    print(f"  Control chars:   {ctrl_zeros:,}  (mostly impossible combos)")
    print(f"  High-byte:       {high_zeros:,}  (UTF-8 internals, encoding noise)")

    if ascii_zeros == 0:
        print(f"\nAll printable ASCII pairs covered. Table is fully converged.")
    elif ascii_zeros < 20:
        print(f"\nNearly converged. {ascii_zeros} exotic printable pairs remain.")
    elif ascii_zeros < 200:
        print(f"\nGood coverage. More data has diminishing returns.")
    else:
        print(f"\n{ascii_zeros} printable pairs unseen. More data would help.")


convergence_analysis(counts, total_bytes)

## 8. Convert to weights

In [ ]:
def counts_to_weights(counts):
    nonzero_mask = counts > 0
    if not nonzero_mask.any():
        return np.full(65536, 65535, dtype=np.uint16)
    log_counts = np.log1p(counts.astype(np.float64))
    max_log = log_counts.max()
    if max_log == 0:
        return np.full(65536, 65535, dtype=np.uint16)
    normalized = log_counts / max_log
    weights = ((1.0 - normalized) * 64900 + 100).astype(np.float64)
    weights[~nonzero_mask] = 65535
    return np.clip(weights, 0, 65535).astype(np.uint16)

weights = counts_to_weights(counts)
print("Weights computed.")

## 9. Diagnostics

In [ ]:
def print_diagnostics(counts, weights, total_bytes, bt):
    if counts.sum() == 0:
        print("NO DATA PROCESSED."); return

    nonzero = counts > 0

    top = np.argsort(counts)[::-1][:15]
    print(f"\nTop 15 most common (of {total_bytes/1e9:.0f} GB):")
    for idx in top:
        b1, b2 = idx >> 8, idx & 0xFF
        c1 = chr(b1) if 32 <= b1 < 127 else f"\\x{b1:02x}"
        c2 = chr(b2) if 32 <= b2 < 127 else f"\\x{b2:02x}"
        print(f"  '{c1}{c2}'  count={counts[idx]:>14,}  weight={weights[idx]:>5}")

    # Boundary decision checks (the ones that matter).
    print(f"\nBoundary decisions (threshold={bt}):")
    checks = [
        # (b1, b2, label, must_be_boundary)
        (ord('q'), ord('z'), 'qz  rare         ', True),
        (ord('x'), ord('j'), 'xj  rare         ', True),
        (ord('z'), ord('x'), 'zx  rare         ', True),
        (ord(' '), ord(' '), '    double space  ', False),
        (ord('r'), ord('e'), 're  return/result ', False),
        (ord('e'), ord('r'), 'er  common suffix ', False),
        (ord('i'), ord('n'), 'in  in/int        ', False),
        (ord('s'), ord('t'), 'st  str/struct    ', False),
        (ord('q'), ord('u'), 'qu  query/queue   ', False),
        (ord('f'), ord('n'), 'fn  Rust keyword  ', False),
    ]
    all_ok = True
    for b1, b2, label, must_boundary in checks:
        idx = (b1 << 8) | b2
        w = weights[idx]
        is_boundary = w >= bt
        ok = is_boundary == must_boundary
        side = "boundary" if is_boundary else "interior"
        print(f"  {label}  w={w:>5}  {side:8s}  {'PASS' if ok else 'FAIL'}")
        if not ok: all_ok = False

    # Spot-check tokenization.
    print(f"\nTokenization spot-checks:")
    def splits(s):
        low = s.lower().encode("ascii")
        pts = [0]
        for i in range(1, len(low)):
            if weights[(low[i-1] << 8) | low[i]] >= bt:
                pts.append(i)
        pts.append(len(low))
        return [s[pts[j]:pts[j+1]] for j in range(len(pts)-1) if pts[j] < pts[j+1]]

    for w in ["parsequery", "HashMap", "function", "return", "struct", "import"]:
        print(f"  '{w}' -> {splits(w)}")

    nz_ok = nonzero.sum() > 15000
    print(f"\nNon-zero pairs: {nonzero.sum():,}  {'PASS' if nz_ok else 'FAIL'}")
    print(f"Boundary checks: {'ALL PASS' if all_ok else 'SOME FAILED'}")


print_diagnostics(counts, weights, total_bytes, BOUNDARY_THRESHOLD)

## 10. Emit `weights.rs`

In [ ]:
def emit_rust_const(weights, total_bytes, dataset_name, output_path):
    if (weights == 65535).all():
        print("Cannot write — no data."); return
    with open(output_path, "w") as f:
        f.write("// Auto-generated by weights_gen_colab.ipynb\n")
        f.write("// Bigram frequency weights for sparse n-gram tokenization.\n")
        f.write("// Rare byte-pairs get high weights (good gram boundaries).\n")
        f.write("// Common byte-pairs get low weights (gram interiors).\n")
        f.write("//\n")
        f.write("// Index: (byte_a << 8) | byte_b\n")
        f.write("// Usage: BIGRAM_WEIGHTS[(b1 as usize) << 8 | (b2 as usize)]\n")
        f.write(f"// Corpus: {total_bytes / 1e9:.1f} GB from {dataset_name}\n")
        f.write("//         (Rust, Python, JS, TS, Go, Java, C, C++, and more)\n")
        f.write("//\n")
        f.write("// DO NOT EDIT BY HAND.\n")
        f.write("\n")
        f.write("/// Pre-trained byte-pair frequency table.\n")
        f.write("pub const BIGRAM_WEIGHTS: [u16; 65536] = [\n")
        for i in range(0, 65536, 16):
            row = ", ".join(f"{weights[i+j]:5}" for j in range(16))
            f.write(f"    {row},\n")
        f.write("];\n")
    print(f"Wrote {output_path} ({os.path.getsize(output_path) / 1024:.0f} KB)")


emit_rust_const(weights, total_bytes, DATASET_NAME, OUTPUT_PATH)

## 11. Download

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_PATH):
        files.download(OUTPUT_PATH)
except ImportError:
    print(f"File at: {OUTPUT_PATH}")

## 12. (Optional) Compare against current weights

In [ ]:
import re as _re

def parse_existing_weights(path):
    if not os.path.exists(path): return None
    with open(path) as f: text = f.read()
    match = _re.search(r'\[([^\]]+)\]', text, _re.DOTALL)
    if not match: return None
    nums = [int(x.strip()) for x in match.group(1).split(',') if x.strip()]
    if len(nums) != 65536: return None
    return np.array(nums, dtype=np.uint16)

def compare_weights(old, new, bt=28000):
    diff = new.astype(np.int32) - old.astype(np.int32)
    changed = np.count_nonzero(diff)
    print(f"Changed: {changed:,} / 65,536 ({changed/655.36:.1f}%)")
    print(f"Max incr: {diff.max():+d}  Max decr: {diff.min():+d}  Mean |diff|: {np.abs(diff).mean():.1f}")

    flipped = (old >= bt) != (new >= bt)
    fc = flipped.sum()
    print(f"Boundary flips: {fc}")
    if fc > 0:
        top = np.argsort(np.abs(diff) * flipped.astype(np.int32))[::-1][:min(fc, 25)]
        for idx in top:
            if not flipped[idx]: continue
            b1, b2 = idx >> 8, idx & 0xFF
            c1 = chr(b1) if 32<=b1<127 else f"\\x{b1:02x}"
            c2 = chr(b2) if 32<=b2<127 else f"\\x{b2:02x}"
            d = "int->bnd" if new[idx]>=bt else "bnd->int"
            print(f"  '{c1}{c2}'  {old[idx]:5}->{new[idx]:5}  ({d})")

# Upload weights_old.rs to /content/, then uncomment:
# old = parse_existing_weights("/content/weights_old.rs")
# if old is not None: compare_weights(old, weights, BOUNDARY_THRESHOLD)
print("Upload weights_old.rs and uncomment above.")

## 13. (Optional) Save raw counts

In [ ]:
np.savez_compressed("/content/bigram_counts_final.npz",
                    counts=counts, total_bytes=np.array(total_bytes))
print(f"Saved ({os.path.getsize('/content/bigram_counts_final.npz')/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download("/content/bigram_counts_final.npz")
except ImportError: pass

---

## After generating

```bash
cp ~/Downloads/weights.rs src/tokenizer/weights.rs
cargo test
cargo bench --bench query_latency -- --sample-size 10
cargo bench --bench index_build -- --sample-size 10
python3 scripts/bench_compare.py --preset react_token_aligned
```

Record before/after in `docs/BENCHMARKS.md`.

To restart: `!rm /content/bigram_checkpoint_v2.npz`